# 3 ACL 离线推理
第 2 章已经生成 OM 模型和输入 BIN。本章先编译 ACL 程序，再完成一次功能检查，最后测量平均时延。

如果仓库不在 CANN Lab 默认目录，请在第一个 Code Cell 中填写 `USER_REPO_ROOT`，或提前设置 `GITCODE_REPO_ROOT`。

In [ ]:
from pathlib import Path
import os, subprocess, sys
USER_REPO_ROOT = ''  # 可选：例如 '/mnt/workspace/my-repo'
DEFAULT_REPO_ROOT = '/mnt/workspace/gitCode/cann/cann-learning-hub'
REPO_ROOT = Path(USER_REPO_ROOT or os.environ.get('GITCODE_REPO_ROOT', DEFAULT_REPO_ROOT))
TUTORIAL_DIR=REPO_ROOT/'reference_practice'/'yolov13_offline_inference'
os.environ['LD_LIBRARY_PATH']=str(Path(sys.prefix)/'lib')+':'+os.environ.get('LD_LIBRARY_PATH','')
MODEL_PATH=TUTORIAL_DIR/'model'/'yolov13.om'; INPUT_PATH=TUTORIAL_DIR/'data'/'bus.bin'; EXECUTABLE=TUTORIAL_DIR/'out'/'yolov13_main'
subprocess.run(['bash','scripts/build.sh'],cwd=TUTORIAL_DIR,check=True)

## 2. 先做一次功能检查
这一 Cell 使用同一份 OM 和输入，只执行一次推理。程序会初始化 ACL、设置设备、加载模型并创建 Dataset，然后调用 `aclmdlExecute`。从 `out` 目录启动是必要的，因为示例内部用 `../src/acl.json` 定位 ACL 配置文件。

推理完成后，程序会解析模型输出并通过 NMS 去除重复检测框。Cell 的输出中会列出每个目标的类别、置信度和检测框坐标，最后汇总识别到的人数和车辆数。检测结果会保存到 `data/bus_result.txt`，供下一章绘制到原图上。

In [ ]:
run=subprocess.run([str(EXECUTABLE),'--model='+str(MODEL_PATH),'--input='+str(INPUT_PATH),'--warmup-runs=0','--runs=1'],cwd=EXECUTABLE.parent,text=True,capture_output=True,check=True)
print(run.stdout); print(run.stderr)

## 3. 按固定次数测量基线
单次结果可以证明流程可用，但不能代表稳定时延。下面先预热 10 次，让设备进入稳定状态，再统计 90 次正式调用；平均值只覆盖 `aclmdlExecute`，不包含模型加载、输入拷贝和后处理。

In [ ]:
run=subprocess.run([str(EXECUTABLE),'--model='+str(MODEL_PATH),'--input='+str(INPUT_PATH),'--warmup-runs=10','--runs=90'],cwd=EXECUTABLE.parent,text=True,capture_output=True,check=True)
print(run.stdout); print(run.stderr)